# Lecture 5: Third-Party Proof-Checking Attestations

Local proof replay can be operationally cumbersome. A specialized provider can run the Lean/Aeneas environment in a controlled setup and publish a signed attestation. This transforms trust in local compilation into trust in a provider, its environment, its signing key custody, and its transparency log.


## Learning Objectives

- Explain the trust transformation from local replay to provider attestation.
- Read a provider attestation.
- Verify an Ed25519 attestation signature.
- Understand why untrusted attestations must score R0.
- Distinguish a provider signature from transparency-log accountability.


## Attestation Contents

A useful attestation records:

- provider identity,
- issue time,
- subject component, repo URL, repo commit, verification dir, kind, backend,
- Lean and lake versions,
- check log and axiom log locations,
- certificate names, statuses, observed axioms, expected axioms,
- provider signature metadata.

The agent must verify both content and trust policy. A valid signature from an untrusted provider is not enough.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.attestation import load_attestation

attestation_path = repo_root / "examples" / "dalek-ed25519.attestation.yaml"
raw = load_attestation(attestation_path)
print(raw.keys())
print(raw["provider"])
print(raw["subject"])
print(raw["certificates"][0])


## Trust Policy

PACTA requires an explicit `--trust-attestation-provider` value. If the attestation provider does not match, the attestation is rejected.

Real attestations should be signed. The included example fixture is unsigned and requires `--allow-unsigned-attestation`, which is suitable only for demos and tests.


In [ ]:
from pacta.attestation import validate_attestation
from pacta.config import RepoConfig

repo = RepoConfig(
    name="dalek-ed25519-verified",
    url="https://github.com/saymrwulf/dalek-ed25519-verified.git",
    kind="ed25519",
    verified_backend="serial/u64",
    certificates=[
        "CurveFieldProofs.fieldImplementation",
        "CurveFieldProofs.edwardsImplementation",
    ],
)

trusted = validate_attestation(
    raw,
    repo,
    path=attestation_path,
    trusted_provider="example-proof-checker.invalid",
    allow_unsigned=True,
)
untrusted = validate_attestation(raw, repo, path=attestation_path)
print("trusted accepted:", trusted.accepted)
print("untrusted accepted:", untrusted.accepted)
print("untrusted diagnostics:", untrusted.diagnostics)


## Provider Threat Model

A proof-checking provider can be valuable, but it introduces new risks:

- It may sign an incorrect result.
- Its environment may be stale or compromised.
- Its signing key may be stolen.
- It may equivocate by showing different results to different agents.
- It may lose log history.

This is why transparency logging matters. A signature says "this provider signed this." A transparency receipt says "this signed result is included in an append-only public structure at this tree head."


## Exercises

- Draw the trusted base for local replay and provider attestation. Mark what changes.
- Explain why a provider attestation must include repo commit, not only repo name.
- Design a monitoring rule that would detect if the provider changes the result for the same commit.
